# PGD-L∞ атака на ResNet18, обученный на CIFAR-10

Kaggle: веса загружаются из `/kaggle/input/models/slomauh/resnet-18-cifar10/pytorch/default/1/resnet18_cifar10_best.pth`, CIFAR-10 — из `/kaggle/input/datasets/oxcdcd/cifar10/cifar10`. Если датасета там нет, он скачивается из интернета.

In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import tarfile
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

SEED = 0
BATCH_SIZE = 256
NUM_WORKERS = 2
EPSILON_NUMERATORS = [0, 2, 4]
PGD_STEPS = 10
EXAMPLES_TO_SHOW = 8
CKPT_PATH = Path('/kaggle/input/models/slomauh/resnet-18-cifar10/pytorch/default/1/resnet18_cifar10_best.pth')
CIFAR_INPUT_PATH = Path('/kaggle/input/datasets/oxcdcd/cifar10/cifar10')
OUTPUT_DIR = Path('/kaggle/working/pgd_results')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Checkpoint

In [ ]:
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f'Файл весов не найден: {CKPT_PATH}')
print('Checkpoint:', CKPT_PATH)

## CIFAR-10 test set

In [ ]:
def locate_cifar10():
    work = Path('/kaggle/working/data')
    work.mkdir(parents=True, exist_ok=True)

    if CIFAR_INPUT_PATH.exists():
        batch_dirs = (
            [CIFAR_INPUT_PATH]
            if CIFAR_INPUT_PATH.name == 'cifar-10-batches-py'
            else list(CIFAR_INPUT_PATH.rglob('cifar-10-batches-py'))
        )
        for source in batch_dirs:
            if source.is_dir():
                destination = work / 'cifar-10-batches-py'
                if not destination.exists():
                    os.symlink(source, destination, target_is_directory=True)
                print('CIFAR-10 загружен из Kaggle Input:', source)
                return work, False

        for archive in CIFAR_INPUT_PATH.rglob('cifar-10-python.tar.gz'):
            if not (work / 'cifar-10-batches-py').exists():
                print('Распаковывается:', archive)
                with tarfile.open(archive) as tar:
                    tar.extractall(work)
            return work, False

        print('По указанному пути нет стандартных файлов CIFAR-10:', CIFAR_INPUT_PATH)
    else:
        print('Путь Kaggle Input не найден:', CIFAR_INPUT_PATH)
    print('CIFAR-10 будет скачан из интернета в /kaggle/working/data.')
    return work, True

imagefolder_test = CIFAR_INPUT_PATH / 'test'
if imagefolder_test.is_dir():
    test_set = torchvision.datasets.ImageFolder(
        root=imagefolder_test, transform=transforms.ToTensor()
    )
    print('CIFAR-10 test загружен как ImageFolder:', imagefolder_test)
    print('Классы ImageFolder:', test_set.class_to_idx)
else:
    DATA_ROOT, NEED_DOWNLOAD = locate_cifar10()
    test_set = torchvision.datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=NEED_DOWNLOAD,
        transform=transforms.ToTensor(),
    )
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False,
    persistent_workers=NUM_WORKERS > 0,
)
CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')
if isinstance(test_set, torchvision.datasets.ImageFolder):
    folder_classes = tuple(test_set.classes)
    numeric_classes = tuple(str(index) for index in range(10))
    if folder_classes not in (CLASSES, numeric_classes):
        raise ValueError(
            f'Неожиданный порядок классов в test/: {folder_classes}. '
            f'Ожидаются названия {CLASSES} либо каталоги 0..9.'
        )
sample_x, sample_y = next(iter(test_loader))
assert tuple(sample_x.shape[1:]) == (3, 32, 32)
assert 0.0 <= sample_x.min().item() and sample_x.max().item() <= 1.0
print(f'Test images: {len(test_set):,}; first batch: {tuple(sample_x.shape)}')

## CIFAR-ResNet18

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std

class NormalizedResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.normalize = Normalize(CIFAR10_MEAN, CIFAR10_STD)
        self.backbone = torchvision.models.resnet18(weights=None, num_classes=10)
        self.backbone.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()

    def forward(self, x):
        return self.backbone(self.normalize(x))

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
state_dict = checkpoint.get('state_dict', checkpoint)

torchvision_state_dict = {}
for key, value in state_dict.items():
    key = key.removeprefix('module.')
    if key.startswith('normalize.'):
        continue
    key = key.replace('linear.', 'fc.')
    key = key.replace('.shortcut.', '.downsample.')
    torchvision_state_dict[key] = value

model = NormalizedResNet18().to(DEVICE)
model.backbone.load_state_dict(torchvision_state_dict, strict=True)
model.eval()
model.requires_grad_(False)

epoch = checkpoint.get('epoch', 'unknown')
val_acc = checkpoint.get('val_acc')
val_text = f'{float(val_acc) * 100:.2f}%' if val_acc is not None else 'unknown'
with torch.no_grad():
    shape_test = model(torch.zeros(4, 3, 32, 32, device=DEVICE))
assert tuple(shape_test.shape) == (4, 10)
print(f'Loaded epoch: {epoch}; validation accuracy: {val_text}')
print('Output shape:', tuple(shape_test.shape))

In [ ]:
@torch.no_grad()
def clean_accuracy(net, loader):
    correct = total = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        correct += (net(images).argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total

baseline_accuracy = clean_accuracy(model, test_loader)
print(f'Clean test accuracy: {baseline_accuracy * 100:.2f}%')

## PGD-L∞

In [ ]:
def pgd_linf(net, images, labels, epsilon, steps=10, alpha=None, random_start=True):
    original = images.detach()
    if epsilon == 0:
        return original.clone()
    if steps <= 0:
        raise ValueError('steps должен быть положительным.')
    alpha = epsilon / 4 if alpha is None else alpha

    if random_start:
        adversarial = original + torch.empty_like(original).uniform_(-epsilon, epsilon)
        adversarial = adversarial.clamp(0.0, 1.0)
    else:
        adversarial = original.clone()

    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = F.cross_entropy(net(adversarial), labels)
        gradient = torch.autograd.grad(loss, adversarial, only_inputs=True)[0]
        with torch.no_grad():
            adversarial = adversarial + alpha * gradient.sign()
            delta = (adversarial - original).clamp(-epsilon, epsilon)
            adversarial = (original + delta).clamp(0.0, 1.0)

    return adversarial.detach()

smoke_images = sample_x[:7].to(DEVICE)
smoke_labels = sample_y[:7].to(DEVICE)
smoke_epsilon = 8 / 255
smoke_adversarial = pgd_linf(model, smoke_images, smoke_labels, smoke_epsilon, PGD_STEPS)
max_delta = (smoke_adversarial - smoke_images).abs().max().item()
assert tuple(smoke_adversarial.shape) == tuple(smoke_images.shape)
assert smoke_adversarial.min().item() >= 0.0 and smoke_adversarial.max().item() <= 1.0
assert max_delta <= smoke_epsilon + 1e-6
assert torch.equal(pgd_linf(model, smoke_images, smoke_labels, 0, PGD_STEPS), smoke_images)
assert all(parameter.grad is None for parameter in model.parameters())
print(f'Smoke-test passed; max L∞ delta = {max_delta * 255:.3f}/255')

## Оценка устойчивости

In [ ]:
def evaluate_attack(net, loader, epsilon, steps, collect_examples=0):
    total = clean_correct = robust_correct = successful = 0
    examples = []
    max_observed_delta = 0.0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        with torch.no_grad():
            clean_predictions = net(images).argmax(1)
        adversarial = pgd_linf(net, images, labels, epsilon, steps)
        with torch.no_grad():
            adversarial_predictions = net(adversarial).argmax(1)

        clean_mask = clean_predictions.eq(labels)
        robust_mask = adversarial_predictions.eq(labels)
        success_mask = clean_mask & ~robust_mask
        total += labels.size(0)
        clean_correct += clean_mask.sum().item()
        robust_correct += robust_mask.sum().item()
        successful += success_mask.sum().item()
        max_observed_delta = max(
            max_observed_delta, (adversarial - images).abs().max().item()
        )

        if len(examples) < collect_examples:
            for index in success_mask.nonzero(as_tuple=False).flatten().tolist():
                examples.append({
                    'clean': images[index].detach().cpu(),
                    'adversarial': adversarial[index].detach().cpu(),
                    'label': labels[index].item(),
                    'clean_prediction': clean_predictions[index].item(),
                    'adversarial_prediction': adversarial_predictions[index].item(),
                })
                if len(examples) == collect_examples:
                    break

    assert max_observed_delta <= epsilon + 1e-6
    return {
        'clean_accuracy': clean_correct / total,
        'robust_accuracy': robust_correct / total,
        'attack_success_rate': successful / clean_correct if clean_correct else 0.0,
        'max_linf': max_observed_delta,
        'examples': examples,
    }

results = []
examples_at_max_epsilon = []
for epsilon_numerator in EPSILON_NUMERATORS:
    epsilon = epsilon_numerator / 255
    started = time.time()
    metrics = evaluate_attack(
        model, test_loader, epsilon, PGD_STEPS,
        collect_examples=EXAMPLES_TO_SHOW if epsilon_numerator == max(EPSILON_NUMERATORS) else 0,
    )
    elapsed = time.time() - started
    results.append({
        'epsilon_numerator': epsilon_numerator,
        'epsilon': epsilon,
        'steps': PGD_STEPS,
        'alpha': epsilon / 4 if epsilon else 0.0,
        'clean_accuracy': metrics['clean_accuracy'],
        'robust_accuracy': metrics['robust_accuracy'],
        'attack_success_rate': metrics['attack_success_rate'],
        'max_linf': metrics['max_linf'],
        'elapsed_seconds': elapsed,
    })
    if epsilon_numerator == max(EPSILON_NUMERATORS):
        examples_at_max_epsilon = metrics['examples']
    print(
        f'ε={epsilon_numerator}/255 | robust accuracy={metrics["robust_accuracy"] * 100:6.2f}% '
        f'| success rate={metrics["attack_success_rate"] * 100:6.2f}% | {elapsed:.1f}s'
    )

assert abs(results[0]['robust_accuracy'] - baseline_accuracy) < 1e-12

## График и примеры

In [ ]:
eps_values = [row['epsilon_numerator'] for row in results]
robust_values = [row['robust_accuracy'] * 100 for row in results]
success_values = [row['attack_success_rate'] * 100 for row in results]

fig, axis = plt.subplots(figsize=(8, 5))
axis.plot(eps_values, robust_values, marker='o', label='Robust accuracy')
axis.plot(eps_values, success_values, marker='s', label='Attack success rate')
axis.set_xlabel('ε (в единицах /255)')
axis.set_ylabel('Процент')
axis.set_title(f'PGD-L∞, {PGD_STEPS} шагов')
axis.set_xticks(eps_values)
axis.set_ylim(0, 100)
axis.grid(alpha=0.3)
axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pgd_robustness_curve.png', dpi=160)
plt.show()

In [ ]:
if not examples_at_max_epsilon:
    print('Для максимального epsilon не найдено успешных атак.')
else:
    rows = len(examples_at_max_epsilon)
    fig, axes = plt.subplots(rows, 3, figsize=(9, 2.8 * rows), squeeze=False)
    epsilon_max = max(EPSILON_NUMERATORS) / 255
    for row, example in enumerate(examples_at_max_epsilon):
        clean = example['clean'].permute(1, 2, 0).numpy()
        adversarial = example['adversarial'].permute(1, 2, 0).numpy()
        perturbation = (adversarial - clean) / (2 * epsilon_max) + 0.5
        axes[row, 0].imshow(clean)
        axes[row, 0].set_title(f'Clean: {CLASSES[example["clean_prediction"]]}')
        axes[row, 1].imshow(adversarial)
        axes[row, 1].set_title(f'PGD: {CLASSES[example["adversarial_prediction"]]}')
        axes[row, 2].imshow(np.clip(perturbation, 0, 1))
        axes[row, 2].set_title('Возмущение (усилено)')
        axes[row, 0].set_ylabel(f'True: {CLASSES[example["label"]]}')
        for axis in axes[row]:
            axis.set_xticks([])
            axis.set_yticks([])
    fig.suptitle(f'Успешные PGD-примеры, ε={max(EPSILON_NUMERATORS)}/255', y=1.0)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'pgd_examples.png', dpi=160, bbox_inches='tight')
    plt.show()

## Сохранение результатов

In [ ]:
csv_path = OUTPUT_DIR / 'pgd_metrics.csv'
json_path = OUTPUT_DIR / 'pgd_metrics.json'
with csv_path.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
with json_path.open('w', encoding='utf-8') as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

print('Kaggle Output:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' ', path)